In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [2]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [3]:
train_dataset = datasets.ImageFolder("balanced_train", transform=transform)
val_dataset = datasets.ImageFolder("test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [4]:
model = models.mobilenet_v3_small(pretrained=True)

model.classifier[3] = nn.Sequential(
    nn.Linear(model.classifier[3].in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 7)
)

model = model.to(device)

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
for param in model.features.parameters():
    param.requires_grad = False

In [6]:
optimizer = optim.AdamW(model.classifier.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

In [7]:
def train_model(epochs):
    best_val = 0

    for epoch in range(epochs):
        print(f"\n===== Epoch {epoch+1}/{epochs} =====")

        model.train()
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_acc = 100 * train_correct / train_total

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total

        print(f"Train Acc: {train_acc:.2f}%")
        print(f"Val Acc:   {val_acc:.2f}%")

        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), "best_emotion_model.pth")
            print("Best model saved")

In [8]:
train_model(epochs=4)


===== Epoch 1/4 =====
Train Acc: 34.80%
Val Acc:   39.40%
Best model saved

===== Epoch 2/4 =====
Train Acc: 38.72%
Val Acc:   40.61%
Best model saved

===== Epoch 3/4 =====
Train Acc: 40.05%
Val Acc:   43.01%
Best model saved

===== Epoch 4/4 =====
Train Acc: 41.11%
Val Acc:   44.18%
Best model saved


In [9]:
for param in model.features.parameters():
    param.requires_grad = True

In [10]:
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [11]:
train_model(epochs=8)


===== Epoch 1/8 =====
Train Acc: 49.13%
Val Acc:   54.56%
Best model saved

===== Epoch 2/8 =====
Train Acc: 55.27%
Val Acc:   56.88%
Best model saved

===== Epoch 3/8 =====
Train Acc: 58.40%
Val Acc:   58.85%
Best model saved

===== Epoch 4/8 =====
Train Acc: 61.04%
Val Acc:   60.28%
Best model saved

===== Epoch 5/8 =====
Train Acc: 62.83%
Val Acc:   60.66%
Best model saved

===== Epoch 6/8 =====
Train Acc: 64.38%
Val Acc:   61.67%
Best model saved

===== Epoch 7/8 =====
Train Acc: 65.92%
Val Acc:   61.09%

===== Epoch 8/8 =====
Train Acc: 67.04%
Val Acc:   62.69%
Best model saved
